In [ ]:
# Import our dependencies

from sklearn.preprocessing import MinMaxScaler

from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dropout
from keras.layers import Dense

In [ ]:
%matplotlib inline

In [ ]:
# Import our dependencies

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
import tensorflow as tf



In [ ]:
# List of tickers
tickers = ['AMZN', 'MSFT', 'AAPL', 'VALE']

# Dictionary to store DataFrames
stock_data = {}

# Loop through each ticker and read the corresponding CSV file
for ticker in tickers:
    ticker_path = Path(f"../Resources/{ticker}.csv")
    stock_data[ticker] = pd.read_csv(ticker_path)
    stock_data[ticker]["Ticker"] = ticker

# Access the DataFrames using the tickers
amzn_stock = stock_data['AMZN']
msft_stock = stock_data['MSFT']
vale_stock = stock_data['VALE']
Apple_stock = stock_data['AAPL']

#Display data for AMZN
print("AMZN Data:")
print(f"MSFT :{len(msft_stock)},\n AMZN:{len(amzn_stock)},\n VALE:{len(vale_stock)},\n AAPL:{len(Apple_stock)} ")

In [ ]:
vale_stock.head()

In [ ]:
vale_stock.iloc[:,2:3].values

In [ ]:
#Using the Open Stock Price to be our traing dataset
training_set=vale_stock.iloc[:,2:3].values

training_set

In [ ]:
#Scalling our Dataset/Normailzing the data
scaler = MinMaxScaler (feature_range=(0,1))
scaled_training_set = scaler.fit_transform(training_set)
scaled_training_set

In [ ]:
#Setting up our features and target variables

X_train = []
y_train = []


for i in range(30,501):
    X_train.append(scaled_training_set[i-30:i,0])
    y_train.append(scaled_training_set[i,0])

    

X_train = np.array(X_train)
y_train = np.array(y_train)

In [ ]:
amzn_stock.shape

In [ ]:
print(X_train.shape)
print(y_train.shape)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
#Reshaping the dataset
X_train=np.reshape(X_train, (X_train.shape[0], X_train.shape[1],1))
X_train.shape

In [ ]:
regressor = Sequential()
regressor

In [ ]:
#Option 1

# Adding Layers into the model 

regressor.add(LSTM(units=50,return_sequences=True,input_shape=(X_train.shape[1], 1)))
# regressor.add(Dropout(0.2))
regressor.add(LSTM(units=50,return_sequences=True))
# regressor.add(Dropout(0.2))

# regressor.add(LSTM(units=50,return_sequences=True))
# regressor.add(Dropout(0.2))

# regressor.add(LSTM(units=50))
# regressor.add(Dropout(0.2))
regressor.add(LSTM(50))
regressor.add(Dense(units=1))

In [ ]:
# # Adding Layers into the model
# regressor.add(LSTM(units=50,return_sequences=True,input_shape=(X_train.shape[1], 1)))
# regressor.add(Dropout(0.2))
# regressor.add(LSTM(units=50,return_sequences=True))
# regressor.add(Dropout(0.2))
# #regressor.add(LSTM(units=35,return_sequences=True))
# #regressor.add(Dropout(0.2))
# regressor.add(LSTM(units=50))
# #regressor.add(Dropout(0.2))
# regressor.add(Dense(units=1))

In [ ]:
regressor.compile(optimizer='adam',loss='mean_squared_error')
regressor.fit(X_train,y_train,epochs=100,batch_size=64, verbose=1)

In [ ]:
Vale_ticker_test = Path("../Resources/VALE_Test.csv")
Vale_stock_Test = pd.read_csv(Vale_ticker_test)
Vale_stock_Test["Ticker"]= "Vale_Test"  

Vale_stock_Test.head()

In [ ]:
#Extracting our 2024 Data for open price
actual_stock_price = Vale_stock_Test.iloc[:,2:3].values
actual_stock_price
# actual_stock_price.shape

In [ ]:
#Preping our Data
dataset_total=pd.concat((vale_stock['Open'],Vale_stock_Test['Open']),axis  =0)
inputs=dataset_total[len(dataset_total)-len(Vale_stock_Test)-30:].values

inputs
inputs = inputs.reshape(-1,1)
inputs= scaler.transform(inputs)


X_test = []

for i in range(30,51):
    X_test.append(inputs[i-30:i,0])
X_test= np.array(X_test)
X_test= np.reshape(X_test,(X_test.shape[0], X_test.shape[1],1))
X_test

actual_stock_price[1]

In [ ]:
X_test[2][-1]

In [ ]:
inputs.shape

In [ ]:
#Predicting the Values for Jan 2024
predicted_stock_price = regressor.predict(X_test)
predicted_stock_price=scaler.inverse_transform(predicted_stock_price)


In [ ]:
predicted_stock_price

In [ ]:
#Plotting the Data

plt.plot(actual_stock_price, color ='red', label ="Actual Vale Open Stock Price")
plt.plot(predicted_stock_price, color ='blue', label ="Predicted Vale Open Stock Price")
plt.title('Vale Stock Price Prediction')
plt.xlabel('Time')
plt.ylabel('Vale Stock Price')
plt.legend()

In [ ]:
predicted_stock_price, actual_stock_price

In [ ]:
Vale_stock_Test["Predicted Stock Price"] = predicted_stock_price
Vale_stock_Test

In [ ]:
predicted_stock_price.shape

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
mse = mean_squared_error(Vale_stock_Test["Open"], predicted_stock_price)
mse

In [ ]:
rmse = np.sqrt(mse)
rmse